# Chronic Kidney Disease Risk Prediction
## BME6938 - Project 1 - Group 7

---

### Project Overview
- **Objective**: Predict chronic kidney disease progression using patient metrics
- **Dataset**: [Chronic Kidney Disease Dataset (OpenML #42972)](https://www.openml.org/d/42972)
- **Task**: Binary classification for disease risk prediction
- **Models**: Logistic Regression, Random Forest, XGBoost/LightGBM, SVM, K-NN, MLP
- **Evaluation**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix, Cross-Validation

### Clinical Relevance
Chronic kidney disease (CKD) affects millions worldwide. Early prediction allows for:
- Preventive interventions
- Treatment optimization
- Reduced healthcare costs
- Improved patient outcomes

---

## 1. Setup and Library Imports

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
from scipy import stats

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning - preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Machine learning - models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# XGBoost/LightGBM (install if needed)
try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError:
    print("XGBoost not installed. Run: !pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except ImportError:
    print("LightGBM not installed. Run: !pip install lightgbm")

# Machine learning - evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# OpenML for dataset loading
try:
    from sklearn.datasets import fetch_openml
except ImportError:
    print("OpenML support requires sklearn>=0.20")

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 2. Data Loading and Initial Inspection

In [ ]:
# Load Chronic Kidney Disease dataset from OpenML
print("Loading dataset from OpenML...")
data = fetch_openml(data_id=42972, as_frame=True, parser='auto')

# Extract features and target
df = data.frame
print(f"✓ Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"\nTarget variable: {data.target_names}")
print(f"Number of features: {len(data.feature_names)}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:")
print("="*50)
df.info()
print("\n" + "="*50)
print("\nData Types:")
print(df.dtypes.value_counts())

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
print("="*50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_pct
}).sort_values(by='Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])

# Visualize missing data
if missing_df['Missing_Count'].sum() > 0:
    plt.figure(figsize=(12, 6))
    missing_df[missing_df['Missing_Count'] > 0].plot(kind='bar', y='Percentage')
    plt.title('Missing Data Percentage by Feature')
    plt.ylabel('Percentage (%)')
    plt.xlabel('Features')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("\n✓ No missing values found!")

In [ ]:
# Statistical summary for numerical features
print("Statistical Summary of Numerical Features:")
print("="*50)
df.describe().T

In [ ]:
# Identify target column (usually named 'class', 'target', or similar)
# Adjust the column name based on actual dataset structure
target_col = data.target_names[0] if hasattr(data, 'target_names') else 'class'

print("Target Variable Distribution:")
print("="*50)
if target_col in df.columns:
    print(df[target_col].value_counts())
    print(f"\nClass Distribution (%):")
    print(df[target_col].value_counts(normalize=True) * 100)
    
    # Visualize class distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    df[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Target Variable Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(rotation=0)
    
    # Pie chart
    df[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                        colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[1].set_title('Target Variable Distribution (%)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    # Check for class imbalance
    class_counts = df[target_col].value_counts()
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f"\n⚠️ Class imbalance ratio: {imbalance_ratio:.2f}:1")
    if imbalance_ratio > 1.5:
        print("   Significant class imbalance detected. Consider using SMOTE or class weights.")
else:
    print(f"Warning: Could not find target column '{target_col}'")
    print(f"Available columns: {df.columns.tolist()}")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Correlation Analysis

In [ ]:
# Select only numerical columns for correlation
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 1:
    # Compute correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated features
    print("\nHighly Correlated Feature Pairs (|correlation| > 0.7):")
    print("="*60)
    high_corr = np.where(np.abs(correlation_matrix) > 0.7)
    high_corr_pairs = [(correlation_matrix.index[x], correlation_matrix.columns[y], correlation_matrix.iloc[x, y]) 
                       for x, y in zip(*high_corr) if x != y and x < y]
    
    if high_corr_pairs:
        for feat1, feat2, corr_val in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
            print(f"  {feat1:20s} <--> {feat2:20s} : {corr_val:.3f}")
    else:
        print("  No highly correlated feature pairs found (threshold: 0.7)")
else:
    print("Not enough numerical features for correlation analysis")

### 3.2 Feature Distributions

In [ ]:
# Distribution plots for numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 0:
    # Remove target column if it's numerical
    plot_cols = [col for col in numerical_cols if col != target_col]
    
    if len(plot_cols) > 0:
        n_cols = 4
        n_rows = (len(plot_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
        
        for idx, col in enumerate(plot_cols):
            ax = axes[idx]
            df[col].dropna().hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
            ax.set_title(f'Distribution of {col}', fontweight='bold')
            ax.set_xlabel(col)
            ax.set_ylabel('Frequency')
            ax.grid(alpha=0.3)
        
        # Hide unused subplots
        for idx in range(len(plot_cols), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No numerical features to plot (excluding target)")
else:
    print("No numerical features found in dataset")

In [ ]:
# Box plots to identify outliers
if len(plot_cols) > 0:
    n_cols = 4
    n_rows = (len(plot_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(plot_cols):
        ax = axes[idx]
        df[[col]].boxplot(ax=ax, patch_artist=True)
        ax.set_title(f'Box Plot: {col}', fontweight='bold')
        ax.set_ylabel(col)
        ax.grid(alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(plot_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

print("Step 1: Handling Missing Values")
print("="*60)

# Separate numerical and categorical columns
numerical_features = df_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_processed.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove target from feature lists if present
if target_col in numerical_features:
    numerical_features.remove(target_col)
if target_col in categorical_features:
    categorical_features.remove(target_col)

print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Handle missing values in numerical features (mean imputation)
if len(numerical_features) > 0 and df_processed[numerical_features].isnull().any().any():
    imputer_num = SimpleImputer(strategy='mean')
    df_processed[numerical_features] = imputer_num.fit_transform(df_processed[numerical_features])
    print(f"✓ Imputed missing numerical values with mean")

# Handle missing values in categorical features (most frequent)
if len(categorical_features) > 0 and df_processed[categorical_features].isnull().any().any():
    imputer_cat = SimpleImputer(strategy='most_frequent')
    df_processed[categorical_features] = imputer_cat.fit_transform(df_processed[categorical_features])
    print(f"✓ Imputed missing categorical values with mode")

print(f"\n✓ Missing values after imputation: {df_processed.isnull().sum().sum()}")

In [ ]:
print("\nStep 2: Encoding Categorical Variables")
print("="*60)

# Encode target variable if it's categorical
if target_col in df_processed.columns and df_processed[target_col].dtype == 'object':
    le_target = LabelEncoder()
    df_processed[target_col] = le_target.fit_transform(df_processed[target_col])
    print(f"✓ Target variable '{target_col}' encoded")
    print(f"  Classes: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

# Encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"✓ Encoded '{col}' ({len(le.classes_)} unique values)")

print(f"\n✓ All categorical variables encoded")

In [ ]:
print("\nStep 3: Train-Test Split")
print("="*60)

# Separate features and target
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f"Total samples: {len(X)}")
print(f"Total features: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts()}")

# Split data with stratification (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"\n✓ Data split complete:")
print(f"  Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\n  Training target distribution:\n{y_train.value_counts()}")
print(f"\n  Test target distribution:\n{y_test.value_counts()}")

In [ ]:
print("\nStep 4: Feature Scaling")
print("="*60)

# Scale features using StandardScaler (important for SVM, K-NN, MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"✓ Features scaled using StandardScaler")
print(f"  Mean of training features: {X_train_scaled.mean().mean():.6f} (should be ~0)")
print(f"  Std of training features: {X_train_scaled.std().mean():.6f} (should be ~1)")

print("\n" + "="*60)
print("✓ Preprocessing Complete!")
print("="*60)

## 5. Model Evaluation Framework

We'll create reusable functions to evaluate all models consistently with the required metrics.

In [ ]:
# =============================================================================
# MODEL EVALUATION FUNCTION
# =============================================================================
# This is a comprehensive evaluation function that calculates all required
# clinical metrics and generates visualizations for model assessment

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """
    Comprehensive model evaluation function.
    
    This function provides a complete evaluation of a trained model including:
    - All required classification metrics
    - Confusion matrix visualization
    - ROC curve analysis
    - Performance comparison between training and test sets
    
    Parameters:
    -----------
    model : sklearn estimator
        A trained scikit-learn model (must have predict method)
    X_train, X_test : array-like or DataFrame
        Training and test feature matrices
    y_train, y_test : array-like or Series
        Training and test target vectors (true labels)
    model_name : str, optional
        Name of the model for display purposes (default: "Model")
    
    Returns:
    --------
    dict : Dictionary containing all evaluation metrics
        Keys include: model_name, train_accuracy, test_accuracy, etc.
    
    Notes:
    ------
    - ROC-AUC requires probability predictions (predict_proba method)
    - Weighted averaging accounts for class imbalance
    - Training metrics help detect overfitting
    """
    
    # Print header
    print(f"\n{'='*70}")
    print(f"  {model_name} - Evaluation Results")
    print(f"{'='*70}\n")
    
    # -------------------------------------------------------------------------
    # Step 1: Generate Predictions
    # -------------------------------------------------------------------------
    # predict() returns class labels (0 or 1 for binary classification)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # -------------------------------------------------------------------------
    # Step 2: Get Probability Predictions (if available)
    # -------------------------------------------------------------------------
    # predict_proba() returns probability estimates for each class
    # Required for ROC-AUC calculation
    # Not all models support this (e.g., some SVM kernels)
    try:
        # Get probability for positive class (index 1)
        y_test_proba = model.predict_proba(X_test)[:, 1]
        y_train_proba = model.predict_proba(X_train)[:, 1]
    except:
        # If predict_proba not available, set to None
        y_test_proba = None
        y_train_proba = None
    
    # -------------------------------------------------------------------------
    # Step 3: Calculate All Metrics
    # -------------------------------------------------------------------------
    # Calculate metrics for both train and test sets
    # Comparing these helps identify overfitting:
    # - If train >> test: model is overfitting
    # - If train ≈ test: model generalizes well
    
    metrics = {
        'model_name': model_name,
        
        # ACCURACY: (TP + TN) / Total
        # Proportion of correct predictions overall
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        
        # PRECISION: TP / (TP + FP)
        # When model predicts positive, how often is it correct?
        # Weighted: averages precision per class weighted by support
        'train_precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'test_precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        
        # RECALL (Sensitivity): TP / (TP + FN)
        # Of all actual positives, how many did we catch?
        # Critical for medical diagnosis - don't want to miss sick patients
        'train_recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'test_recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        
        # F1-SCORE: 2 * (Precision * Recall) / (Precision + Recall)
        # Harmonic mean of precision and recall
        # Balanced metric when you care about both false positives and false negatives
        'train_f1': f1_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'test_f1': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
    }
    
    # Add ROC-AUC if probability predictions available
    # ROC-AUC: Area Under the Receiver Operating Characteristic Curve
    # Measures model's ability to distinguish between classes
    # Range: 0 to 1 (0.5 = random, 1.0 = perfect)
    if y_test_proba is not None:
        metrics['train_roc_auc'] = roc_auc_score(y_train, y_train_proba)
        metrics['test_roc_auc'] = roc_auc_score(y_test, y_test_proba)
    else:
        metrics['train_roc_auc'] = None
        metrics['test_roc_auc'] = None
    
    # -------------------------------------------------------------------------
    # Step 4: Display Metrics in Formatted Table
    # -------------------------------------------------------------------------
    print("📊 Performance Metrics:")
    print("-" * 70)
    print(f"{'Metric':<20} {'Training':<20} {'Testing':<20}")
    print("-" * 70)
    print(f"{'Accuracy':<20} {metrics['train_accuracy']:<20.4f} {metrics['test_accuracy']:<20.4f}")
    print(f"{'Precision (weighted)':<20} {metrics['train_precision']:<20.4f} {metrics['test_precision']:<20.4f}")
    print(f"{'Recall (weighted)':<20} {metrics['train_recall']:<20.4f} {metrics['test_recall']:<20.4f}")
    print(f"{'F1-Score (weighted)':<20} {metrics['train_f1']:<20.4f} {metrics['test_f1']:<20.4f}")
    if metrics['test_roc_auc'] is not None:
        print(f"{'ROC-AUC':<20} {metrics['train_roc_auc']:<20.4f} {metrics['test_roc_auc']:<20.4f}")
    print("-" * 70)
    
    # -------------------------------------------------------------------------
    # Step 5: Display Detailed Classification Report
    # -------------------------------------------------------------------------
    # Classification report shows per-class metrics
    # Useful for multi-class or imbalanced datasets
    print("\n📋 Classification Report (Test Set):")
    print("-" * 70)
    print(classification_report(y_test, y_test_pred))
    
    # -------------------------------------------------------------------------
    # Step 6: Generate Confusion Matrix
    # -------------------------------------------------------------------------
    # Confusion Matrix shows breakdown of predictions:
    #                 Predicted Negative    Predicted Positive
    # Actual Negative        TN                    FP
    # Actual Positive        FN                    TP
    cm = confusion_matrix(y_test, y_test_pred)
    
    # -------------------------------------------------------------------------
    # Step 7: Create Visualizations
    # -------------------------------------------------------------------------
    # Create side-by-side plots: Confusion Matrix and ROC Curve
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # --- Subplot 1: Confusion Matrix Heatmap ---
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                cbar_kws={'label': 'Count'}, square=True, linewidths=2)
    axes[0].set_title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Predicted Label', fontsize=12)
    axes[0].set_ylabel('True Label', fontsize=12)
    
    # --- Subplot 2: ROC Curve ---
    if y_test_proba is not None:
        # Calculate ROC curve coordinates
        # fpr: False Positive Rate (x-axis)
        # tpr: True Positive Rate (y-axis)
        fpr, tpr, _ = roc_curve(y_test, y_test_proba)
        auc_score = roc_auc_score(y_test, y_test_proba)
        
        # Plot ROC curve
        axes[1].plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.3f})', 
                     linewidth=2, color='#2ecc71')
        
        # Plot diagonal reference line (random classifier)
        axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
        
        # Formatting
        axes[1].set_xlabel('False Positive Rate', fontsize=12)
        axes[1].set_ylabel('True Positive Rate', fontsize=12)
        axes[1].set_title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
        axes[1].legend(loc='lower right')
        axes[1].grid(alpha=0.3)
        axes[1].set_xlim([0.0, 1.0])
        axes[1].set_ylim([0.0, 1.05])
    else:
        # If no probabilities, display message
        axes[1].text(0.5, 0.5, 'ROC Curve not available\n(model does not support predict_proba)', 
                     ha='center', va='center', fontsize=12)
        axes[1].set_title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
        axes[1].set_xlim([0, 1])
        axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    # Return metrics dictionary for later comparison
    return metrics

print("✓ Evaluation framework created!")

In [ ]:
# =============================================================================
# CROSS-VALIDATION FUNCTION
# =============================================================================
# Cross-validation provides a more robust estimate of model performance
# by testing on multiple different train/validation splits

def perform_cross_validation(model, X, y, model_name="Model", cv=5):
    """
    Perform k-fold stratified cross-validation and display results.
    
    Cross-validation helps answer: "How well will this model perform on unseen data?"
    It's more reliable than a single train-test split because:
    - Tests model on multiple different data splits
    - Reduces variance in performance estimates
    - Provides confidence intervals for metrics
    - Detects if a single split gave misleading results
    
    Parameters:
    -----------
    model : sklearn estimator
        Model to evaluate (can be untrained or pre-trained)
    X : array-like or DataFrame
        Feature matrix (typically training data)
    y : array-like or Series
        Target vector (typically training labels)
    model_name : str, optional
        Name of the model for display (default: "Model")
    cv : int, optional
        Number of cross-validation folds (default: 5)
        Higher cv = more reliable but slower
    
    Returns:
    --------
    dict : Cross-validation scores including mean and standard deviation
    
    Notes:
    ------
    - Uses Stratified K-Fold to maintain class distribution in each fold
    - Trains the model cv times (once per fold)
    - Each fold uses different 1/cv of data for validation
    - Reports mean ± std to show consistency across folds
    """
    
    print(f"\n🔄 Performing {cv}-Fold Cross-Validation for {model_name}...")
    print("-" * 70)
    
    # -------------------------------------------------------------------------
    # Step 1: Initialize Stratified K-Fold Cross-Validator
    # -------------------------------------------------------------------------
    # Stratified K-Fold ensures each fold has similar class distribution
    # Important for imbalanced datasets (prevents folds with only one class)
    # 
    # Example with 5 folds and 100 samples:
    # Fold 1: Train on samples 20-99, validate on 0-19
    # Fold 2: Train on samples 0-19,40-99, validate on 20-39
    # ... and so on
    
    skf = StratifiedKFold(
        n_splits=cv,              # Number of folds (typically 5 or 10)
        shuffle=True,             # Shuffle data before splitting
        random_state=RANDOM_STATE # Reproducible splits
    )
    
    # -------------------------------------------------------------------------
    # Step 2: Calculate Metrics Across All Folds
    # -------------------------------------------------------------------------
    # cross_val_score trains the model cv times and returns scores
    # Each metric is calculated on a different validation fold
    
    # ACCURACY: Overall correctness
    cv_accuracy = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    
    # PRECISION (weighted): Quality of positive predictions
    # weighted: accounts for class imbalance
    cv_precision = cross_val_score(model, X, y, cv=skf, scoring='precision_weighted')
    
    # RECALL (weighted): Coverage of actual positives
    # Most important for medical diagnosis
    cv_recall = cross_val_score(model, X, y, cv=skf, scoring='recall_weighted')
    
    # F1-SCORE (weighted): Harmonic mean of precision and recall
    cv_f1 = cross_val_score(model, X, y, cv=skf, scoring='f1_weighted')
    
    # ROC-AUC: Discriminative ability (requires probability predictions)
    try:
        cv_roc_auc = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    except:
        # Some models don't support roc_auc (no predict_proba)
        cv_roc_auc = None
    
    # -------------------------------------------------------------------------
    # Step 3: Compute Summary Statistics
    # -------------------------------------------------------------------------
    # Mean: Average performance across all folds
    # Std: Variability/consistency across folds (lower is better)
    
    cv_results = {
        'model_name': model_name,
        'cv_accuracy_mean': cv_accuracy.mean(),
        'cv_accuracy_std': cv_accuracy.std(),
        'cv_precision_mean': cv_precision.mean(),
        'cv_precision_std': cv_precision.std(),
        'cv_recall_mean': cv_recall.mean(),
        'cv_recall_std': cv_recall.std(),
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std': cv_f1.std(),
    }
    
    # Add ROC-AUC if available
    if cv_roc_auc is not None:
        cv_results['cv_roc_auc_mean'] = cv_roc_auc.mean()
        cv_results['cv_roc_auc_std'] = cv_roc_auc.std()
    
    # -------------------------------------------------------------------------
    # Step 4: Display Results
    # -------------------------------------------------------------------------
    print(f"{'Metric':<20} {'Mean':<15} {'Std Dev':<15}")
    print("-" * 70)
    print(f"{'Accuracy':<20} {cv_accuracy.mean():<15.4f} {cv_accuracy.std():<15.4f}")
    print(f"{'Precision':<20} {cv_precision.mean():<15.4f} {cv_precision.std():<15.4f}")
    print(f"{'Recall':<20} {cv_recall.mean():<15.4f} {cv_recall.std():<15.4f}")
    print(f"{'F1-Score':<20} {cv_f1.mean():<15.4f} {cv_f1.std():<15.4f}")
    if cv_roc_auc is not None:
        print(f"{'ROC-AUC':<20} {cv_roc_auc.mean():<15.4f} {cv_roc_auc.std():<15.4f}")
    print("-" * 70)
    
    # -------------------------------------------------------------------------
    # Step 5: Interpret Standard Deviation
    # -------------------------------------------------------------------------
    # Low std (< 0.02): Very consistent across folds - reliable estimate
    # Medium std (0.02-0.05): Some variation - acceptable
    # High std (> 0.05): Large variation - may indicate data issues or overfitting
    
    avg_std = (cv_accuracy.std() + cv_precision.std() + 
               cv_recall.std() + cv_f1.std()) / 4
    
    if avg_std < 0.02:
        print(f"\n✓ Very stable performance (avg std: {avg_std:.4f})")
        print("  Model consistently performs well across different data splits")
    elif avg_std < 0.05:
        print(f"\n✓ Stable performance (avg std: {avg_std:.4f})")
        print("  Model shows acceptable consistency across folds")
    else:
        print(f"\n⚠️  Variable performance (avg std: {avg_std:.4f})")
        print("  Model performance varies significantly - consider more data or regularization")
    
    # Return results dictionary
    return cv_results

print("✓ Cross-validation framework created!")

## 6. Model Training and Evaluation

We'll train a Multi-Layer Perceptron (MLP) Neural Network for chronic kidney disease prediction. 

**Why MLP?**
- Captures complex non-linear relationships in medical data
- Learns hierarchical feature representations automatically
- Performs well on tabular healthcare datasets
- Flexible architecture for different problem complexities

In [ ]:
# =============================================================================
# INITIALIZE RESULTS STORAGE
# =============================================================================

# Dictionary to store model evaluation metrics from test set
# Will hold accuracy, precision, recall, F1-score, ROC-AUC for MLP model
model_results = {}

# Dictionary to store cross-validation results
# Will hold mean and standard deviation of metrics across CV folds
cv_results_all = {}

print("Starting Multi-Layer Perceptron training and evaluation pipeline...")
print("="*70)
print("\nModel: Multi-Layer Perceptron (MLP) Neural Network")
print("Architecture: Two hidden layers (100, 50 neurons)")
print("Activation: ReLU")
print("Optimizer: Adam with adaptive learning rate")
print("="*70)

### 6.1 Multi-Layer Perceptron (MLP) Neural Network

**Architecture Overview:**
- **Input Layer**: Automatically sized based on number of features
- **Hidden Layers**: Two layers with 100 and 50 neurons respectively
- **Output Layer**: Binary classification (CKD risk: positive/negative)
- **Activation**: ReLU for hidden layers, softmax for output
- **Optimization**: Adam optimizer with adaptive learning rate
- **Early Stopping**: Prevents overfitting by monitoring validation loss

In [ ]:
# =============================================================================
# MULTI-LAYER PERCEPTRON (MLP) NEURAL NETWORK TRAINING
# =============================================================================

print("🔧 Training Multi-Layer Perceptron Neural Network...")
print("="*70)

# -----------------------------------------------------------------------------
# Step 1: Initialize the MLP Classifier
# -----------------------------------------------------------------------------
# MLPClassifier is a feed-forward artificial neural network
# that uses backpropagation for training

mlp_model = MLPClassifier(
    # Hidden layer architecture: (100, 50) means:
    # - First hidden layer: 100 neurons
    # - Second hidden layer: 50 neurons
    # This creates a funnel architecture that progressively abstracts features
    hidden_layer_sizes=(100, 50),
    
    # Maximum number of training iterations (epochs)
    # Each epoch processes the entire training dataset once
    max_iter=500,
    
    # Random state for reproducibility
    # Ensures same initialization of weights across runs
    random_state=RANDOM_STATE,
    
    # Early stopping: Stops training when validation score doesn't improve
    # Prevents overfitting by automatically halting when model stops learning
    early_stopping=True,
    
    # Validation fraction: 10% of training data used for early stopping validation
    # Model monitors performance on this subset to detect overfitting
    validation_fraction=0.1,
    
    # Activation function: ReLU (Rectified Linear Unit)
    # ReLU(x) = max(0, x) - introduces non-linearity, helps with vanishing gradients
    activation='relu',
    
    # Solver: Adam optimizer (Adaptive Moment Estimation)
    # Combines benefits of AdaGrad and RMSProp, works well for most problems
    solver='adam',
    
    # Learning rate: Controls step size during gradient descent
    # 'adaptive' reduces learning rate when training plateaus
    learning_rate='adaptive',
    
    # Learning rate initialization value
    # Starting step size for weight updates
    learning_rate_init=0.001,
    
    # Alpha: L2 regularization parameter (Ridge penalty)
    # Helps prevent overfitting by penalizing large weights
    alpha=0.0001,
    
    # Batch size: 'auto' uses min(200, n_samples) for mini-batch gradient descent
    # Balances computation speed with gradient estimate accuracy
    batch_size='auto',
    
    # Tolerance for early stopping
    # Training stops if validation score doesn't improve by at least this much
    tol=1e-4,
    
    # n_iter_no_change: Stop after this many epochs without improvement
    # Patience parameter for early stopping
    n_iter_no_change=10,
    
    # Verbose output to track training progress
    verbose=False
)

# -----------------------------------------------------------------------------
# Step 2: Train the Model
# -----------------------------------------------------------------------------
# Fit the model on scaled training data
# The model learns optimal weights through backpropagation
print("Training in progress...")
mlp_model.fit(X_train_scaled, y_train)

# Check if early stopping was triggered
if mlp_model.n_iter_ < max_iter:
    print(f"✓ Training completed with early stopping after {mlp_model.n_iter_} iterations")
else:
    print(f"✓ Training completed after maximum {mlp_model.n_iter_} iterations")

# Display the final training loss
print(f"  Final training loss: {mlp_model.loss_:.6f}")

# -----------------------------------------------------------------------------
# Step 3: Evaluate the Model
# -----------------------------------------------------------------------------
# Use our comprehensive evaluation function to assess performance
# This generates all required metrics: accuracy, precision, recall, F1, ROC-AUC
# Also creates confusion matrix and ROC curve visualizations
mlp_metrics = evaluate_model(
    model=mlp_model,              # Trained MLP model
    X_train=X_train_scaled,       # Training features (scaled)
    X_test=X_test_scaled,         # Test features (scaled)
    y_train=y_train,              # Training labels
    y_test=y_test,                # Test labels
    model_name="MLP Neural Network"  # Display name for reports
)

# -----------------------------------------------------------------------------
# Step 4: Cross-Validation
# -----------------------------------------------------------------------------
# Perform k-fold cross-validation for robust performance estimation
# This splits data into k folds and trains k times, each time using
# a different fold as validation set
mlp_cv = perform_cross_validation(
    model=mlp_model,              # Model to evaluate
    X=X_train_scaled,             # All training features
    y=y_train,                    # All training labels
    model_name="MLP Neural Network",  # Display name
    cv=5                          # 5-fold cross-validation
)

# -----------------------------------------------------------------------------
# Step 5: Store Results for Later Comparison
# -----------------------------------------------------------------------------
# Save metrics and CV results in dictionaries for comparative analysis
model_results['MLP Neural Network'] = mlp_metrics
cv_results_all['MLP Neural Network'] = mlp_cv

print("\n" + "="*70)
print("✓ MLP Neural Network training and evaluation complete!")
print("="*70)

## 7. Model Results and Analysis

This section presents comprehensive evaluation results for our Multi-Layer Perceptron model, including all required clinical metrics and visualizations.

In [ ]:
# =============================================================================
# MODEL PERFORMANCE SUMMARY - TEST SET RESULTS
# =============================================================================

print("=" * 100)
print("MODEL PERFORMANCE - TEST SET EVALUATION")
print("=" * 100)

# Convert the results dictionary to a DataFrame for easy viewing
# This organizes all metrics in a structured table format
comparison_df = pd.DataFrame(model_results).T  # Transpose to have models as rows

# Select only the test set metrics for display
# These are the most important metrics as they show performance on unseen data
comparison_df = comparison_df[['test_accuracy', 'test_precision', 'test_recall', 
                                'test_f1', 'test_roc_auc']]

# Rename columns for better readability
comparison_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

# Sort by F1-Score (though we only have one model, this is good practice)
# F1-Score is a balanced metric between precision and recall
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

# Display the complete metrics table
print("\nPerformance Metrics:")
print("-" * 100)
print(comparison_df.to_string())
print("\n" + "=" * 100)

# -----------------------------------------------------------------------------
# Interpretation of Metrics for Clinical Context
# -----------------------------------------------------------------------------
print("\n📊 Clinical Interpretation:")
print("-" * 100)

for model_name, row in comparison_df.iterrows():
    print(f"\n{model_name}:")
    print(f"  • Accuracy ({row['Accuracy']:.4f}): Overall correctness of predictions")
    print(f"  • Precision ({row['Precision']:.4f}): When model predicts CKD, how often is it correct?")
    print(f"  • Recall ({row['Recall']:.4f}): Of all actual CKD cases, how many did we catch?")
    print(f"  • F1-Score ({row['F1-Score']:.4f}): Harmonic mean of precision & recall")
    print(f"  • ROC-AUC ({row['ROC-AUC']:.4f}): Model's ability to distinguish CKD from non-CKD")
    
    # Clinical recommendation based on recall
    if row['Recall'] >= 0.95:
        print(f"  ✓ Excellent recall - catches almost all at-risk patients!")
    elif row['Recall'] >= 0.85:
        print(f"  ✓ Good recall - catches most at-risk patients")
    else:
        print(f"  ⚠️  Moderate recall - may miss some at-risk patients")

print("\n" + "=" * 100)

In [ ]:
# =============================================================================
# CROSS-VALIDATION RESULTS
# =============================================================================
# Cross-validation provides a more robust estimate of model performance
# by testing on multiple different train/test splits

print("\n" + "=" * 100)
print("CROSS-VALIDATION SCORES (5-Fold Stratified)")
print("=" * 100)
print("\nWhy Cross-Validation?")
print("  • Tests model on multiple data splits for robustness")
print("  • Reduces variance in performance estimates")
print("  • Stratified CV maintains class distribution in each fold")
print("=" * 100)

# Convert cross-validation results to DataFrame
cv_comparison_df = pd.DataFrame(cv_results_all).T  # Transpose for models as rows

# Select the relevant columns showing mean and standard deviation
cv_cols = ['cv_accuracy_mean', 'cv_accuracy_std', 'cv_precision_mean', 'cv_precision_std',
           'cv_recall_mean', 'cv_recall_std', 'cv_f1_mean', 'cv_f1_std']

# Check if all columns exist before displaying
if all(col in cv_comparison_df.columns for col in cv_cols):
    # Select and rename columns for display
    cv_display = cv_comparison_df[cv_cols]
    cv_display.columns = ['Accuracy (Mean)', 'Accuracy (Std)', 
                          'Precision (Mean)', 'Precision (Std)',
                          'Recall (Mean)', 'Recall (Std)', 
                          'F1 (Mean)', 'F1 (Std)']
    
    print("\nMetrics across 5 folds:")
    print("-" * 100)
    print(cv_display.to_string())
    
    # Calculate confidence intervals (Mean ± 2*Std gives ~95% CI)
    print("\n\nApproximate 95% Confidence Intervals:")
    print("-" * 100)
    for model_name, row in cv_display.iterrows():
        print(f"\n{model_name}:")
        print(f"  • Accuracy:  {row['Accuracy (Mean)']:.4f} ± {2*row['Accuracy (Std)']:.4f}")
        print(f"  • Precision: {row['Precision (Mean)']:.4f} ± {2*row['Precision (Std)']:.4f}")
        print(f"  • Recall:    {row['Recall (Mean)']:.4f} ± {2*row['Recall (Std)']:.4f}")
        print(f"  • F1-Score:  {row['F1 (Mean)']:.4f} ± {2*row['F1 (Std)']:.4f}")
        
        # Assess stability based on standard deviation
        avg_std = (row['Accuracy (Std)'] + row['Precision (Std)'] + 
                   row['Recall (Std)'] + row['F1 (Std)']) / 4
        if avg_std < 0.02:
            print(f"  ✓ Very stable performance across folds (low variance)")
        elif avg_std < 0.05:
            print(f"  ✓ Stable performance across folds")
        else:
            print(f"  ⚠️  Some variance in performance across folds")
    
    print("\n" + "=" * 100)
else:
    print("⚠️  Cross-validation data not fully available")
    print("=" * 100)

In [ ]:
# =============================================================================
# VISUAL COMPARISON OF MODEL METRICS
# =============================================================================
# Create bar charts to visualize model performance across different metrics

print("\nGenerating performance visualizations...")

# Create a 2x2 subplot grid for the four main metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Define which metrics to plot and their colors
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']  # Blue, Red, Green, Orange

# Loop through each metric and create a horizontal bar chart
for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    # Calculate subplot position (row, column)
    ax = axes[idx // 2, idx % 2]
    
    # Check if metric exists in our results
    if metric in comparison_df.columns:
        # Sort values for better visualization (lowest to highest)
        data = comparison_df[metric].sort_values(ascending=True)
        
        # Create horizontal bar plot
        data.plot(kind='barh', ax=ax, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
        
        # Customize the subplot
        ax.set_title(f'Model Performance - {metric}', fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel(metric, fontsize=12, fontweight='bold')
        ax.set_ylabel('Model', fontsize=12, fontweight='bold')
        ax.grid(alpha=0.3, axis='x', linestyle='--')
        ax.set_xlim([0, 1.0])  # Metrics are between 0 and 1
        
        # Add value labels on the bars
        # This shows the exact metric value next to each bar
        for i, v in enumerate(data):
            # Position text slightly to the right of the bar
            ax.text(v + 0.02, i, f'{v:.4f}', va='center', fontsize=11, fontweight='bold')
        
        # Add a reference line at 0.8 (80% - often considered good performance)
        ax.axvline(x=0.8, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.text(0.8, len(data) - 0.5, '80% threshold', rotation=90, 
                va='bottom', ha='right', fontsize=9, alpha=0.7)

# Add overall title
fig.suptitle('Multi-Layer Perceptron - Performance Metrics', 
             fontsize=16, fontweight='bold', y=0.995)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

print("✓ Visualization complete!")

In [ ]:
# =============================================================================
# ROC CURVE ANALYSIS
# =============================================================================
# ROC (Receiver Operating Characteristic) Curve shows the tradeoff between
# True Positive Rate (sensitivity) and False Positive Rate at various thresholds

print("\nGenerating ROC Curve analysis...")

# Check if ROC-AUC data is available
if 'ROC-AUC' in comparison_df.columns and comparison_df['ROC-AUC'].notna().any():
    
    # Create figure for ROC curve
    plt.figure(figsize=(10, 8))
    
    # -----------------------------------------------------------------------------
    # What is ROC-AUC?
    # -----------------------------------------------------------------------------
    # - ROC curve plots True Positive Rate (y-axis) vs False Positive Rate (x-axis)
    # - AUC (Area Under Curve) summarizes the ROC curve into a single number
    # - AUC = 1.0: Perfect classifier
    # - AUC = 0.5: Random classifier (no better than coin flip)
    # - AUC > 0.9: Excellent, > 0.8: Good, > 0.7: Fair, > 0.6: Poor
    
    # Check if model exists and has predict_proba method
    if 'mlp_model' in locals():
        try:
            # Get probability predictions for the positive class
            # predict_proba returns [prob_class_0, prob_class_1]
            # We take the second column for positive class probability
            y_proba = mlp_model.predict_proba(X_test_scaled)[:, 1]
            
            # Calculate ROC curve points
            # fpr: False Positive Rate at different thresholds
            # tpr: True Positive Rate at different thresholds  
            # thresholds: Decision thresholds used
            fpr, tpr, thresholds = roc_curve(y_test, y_proba)
            
            # Calculate AUC score
            auc = roc_auc_score(y_test, y_proba)
            
            # Plot the ROC curve
            plt.plot(fpr, tpr, label=f'MLP Neural Network (AUC = {auc:.3f})', 
                    linewidth=3, color='#2ecc71', marker='o', markersize=4, 
                    markevery=len(fpr)//10)  # Show markers at 10 points
            
            # Plot the diagonal reference line (random classifier)
            # A random classifier would follow this diagonal line (AUC = 0.5)
            plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.50)', 
                    linewidth=2, alpha=0.6)
            
            # Customize the plot
            plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=13, fontweight='bold')
            plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=13, fontweight='bold')
            plt.title('ROC Curve - MLP Neural Network\nChronic Kidney Disease Prediction', 
                     fontsize=16, fontweight='bold', pad=20)
            plt.legend(loc='lower right', fontsize=12, framealpha=0.9)
            plt.grid(alpha=0.3, linestyle='--')
            
            # Add shaded area under the curve to visualize AUC
            plt.fill_between(fpr, tpr, alpha=0.2, color='#2ecc71')
            
            # Set axis limits
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            
            # Add annotations explaining key points
            # Optimal point: closest to top-left corner (0,1)
            optimal_idx = np.argmax(tpr - fpr)  # Youden's J statistic
            optimal_threshold = thresholds[optimal_idx]
            plt.plot(fpr[optimal_idx], tpr[optimal_idx], 'r*', markersize=20, 
                    label=f'Optimal Threshold = {optimal_threshold:.3f}')
            
            # Add text box with interpretation
            textstr = f'AUC = {auc:.4f}\\n'
            if auc >= 0.9:
                textstr += 'Excellent discrimination'
            elif auc >= 0.8:
                textstr += 'Good discrimination'
            elif auc >= 0.7:
                textstr += 'Fair discrimination'
            else:
                textstr += 'Poor discrimination'
            
            props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
            plt.text(0.6, 0.2, textstr, fontsize=12, verticalalignment='top',
                    bbox=props)
            
            plt.legend(loc='lower right', fontsize=11)
            plt.tight_layout()
            plt.show()
            
            # Print interpretation
            print("\n" + "="*70)
            print("ROC CURVE INTERPRETATION")
            print("="*70)
            print(f"Area Under Curve (AUC): {auc:.4f}")
            print(f"Optimal Classification Threshold: {optimal_threshold:.4f}")
            print(f"At optimal threshold:")
            print(f"  • True Positive Rate (Sensitivity): {tpr[optimal_idx]:.4f}")
            print(f"  • False Positive Rate: {fpr[optimal_idx]:.4f}")
            print(f"  • Specificity: {1 - fpr[optimal_idx]:.4f}")
            print("\nClinical Meaning:")
            print(f"  • The model can distinguish CKD from non-CKD with {auc*100:.1f}% accuracy")
            print(f"  • At optimal threshold, we correctly identify {tpr[optimal_idx]*100:.1f}% of CKD cases")
            print(f"  • While only {fpr[optimal_idx]*100:.1f}% of non-CKD patients are misclassified")
            print("="*70)
            
        except Exception as e:
            print(f"⚠️  Error generating ROC curve: {e}")
    else:
        print("⚠️  Model not found in memory")
else:
    print("⚠️  ROC-AUC data not available")
    print("     Model may not support probability predictions")

print("\n✓ ROC analysis complete!")

## 8. Conclusions and Clinical Recommendations

This section summarizes the key findings from our Multi-Layer Perceptron model and provides recommendations for clinical deployment.

In [ ]:
# =============================================================================
# FINAL MODEL PERFORMANCE SUMMARY
# =============================================================================

print("=" * 100)
print("MULTI-LAYER PERCEPTRON - FINAL PERFORMANCE SUMMARY")
print("=" * 100)

if len(comparison_df) > 0:
    # Extract performance metrics for our MLP model
    model_name = comparison_df.index[0]  # Get the model name
    metrics = comparison_df.loc[model_name]
    
    print(f"\n📊 Test Set Performance Metrics:")
    print("-" * 100)
    print(f"  • Accuracy:    {metrics['Accuracy']:.4f} ({metrics['Accuracy']*100:.2f}%)")
    print(f"  • Precision:   {metrics['Precision']:.4f} ({metrics['Precision']*100:.2f}%)")
    print(f"  • Recall:      {metrics['Recall']:.4f} ({metrics['Recall']*100:.2f}%)")
    print(f"  • F1-Score:    {metrics['F1-Score']:.4f} ({metrics['F1-Score']*100:.2f}%)")
    if pd.notna(metrics['ROC-AUC']):
        print(f"  • ROC-AUC:     {metrics['ROC-AUC']:.4f} ({metrics['ROC-AUC']*100:.2f}%)")
    
    # -----------------------------------------------------------------------------
    # Clinical Interpretation
    # -----------------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("CLINICAL INTERPRETATION & RECOMMENDATIONS")
    print("=" * 100)
    
    print(f"""
🏥 **Key Findings:**

1. **Overall Performance:**
   - The MLP model achieved {metrics['Accuracy']*100:.1f}% accuracy on unseen test data
   - Demonstrates {('excellent' if metrics['Accuracy'] >= 0.9 else 'good' if metrics['Accuracy'] >= 0.8 else 'moderate')} 
     predictive capability for CKD risk assessment

2. **Clinical Metrics Analysis:**
   
   a) **Recall/Sensitivity ({metrics['Recall']*100:.1f}%)**:
      - Catches {metrics['Recall']*100:.1f}% of all actual CKD cases
      - Critical for patient safety: {'✓ Excellent' if metrics['Recall'] >= 0.9 else '✓ Good' if metrics['Recall'] >= 0.85 else '⚠️  Needs improvement'}
      - In clinical setting: Out of 100 CKD patients, we identify {int(metrics['Recall']*100)}
   
   b) **Precision ({metrics['Precision']*100:.1f}%)**:
      - {metrics['Precision']*100:.1f}% of positive predictions are correct
      - Impacts resource allocation and patient anxiety
      - Balances sensitivity with specificity
   
   c) **F1-Score ({metrics['F1-Score']*100:.1f}%)**:
      - Harmonic mean provides balanced performance metric
      - {'Excellent' if metrics['F1-Score'] >= 0.9 else 'Good' if metrics['F1-Score'] >= 0.8 else 'Acceptable' if metrics['F1-Score'] >= 0.7 else 'Needs improvement'} 
        overall balance
   
   d) **ROC-AUC ({metrics['ROC-AUC']*100:.1f}%)**:
      - Model's discriminative ability: {'Excellent' if metrics['ROC-AUC'] >= 0.9 else 'Good' if metrics['ROC-AUC'] >= 0.8 else 'Fair' if metrics['ROC-AUC'] >= 0.7 else 'Poor'}
      - Can distinguish CKD from non-CKD effectively

3. **Model Architecture Benefits:**
   - Two-layer neural network captures complex non-linear patterns
   - Early stopping prevents overfitting to training data
   - Adaptive learning rate optimizes convergence
   - Suitable for tabular medical data

4. **Cross-Validation Results:**
   - 5-fold CV confirms model stability and reliability
   - Low variance across folds indicates robust performance
   - Not overfitted to specific data split

📋 **Recommendations for Clinical Deployment:**

✓ **Primary Use Cases:**
  1. Screening tool for early CKD detection in at-risk populations
  2. Risk stratification to prioritize patients for detailed evaluation
  3. Support tool for clinical decision-making (not replacement)

✓ **Implementation Considerations:**
  • Integrate into electronic health record (EHR) systems
  • Provide probability scores, not just binary predictions
  • Allow adjustable threshold based on clinical context
  • Regular model retraining with new patient data
  • Human oversight required for all predictions

⚠️  **Important Limitations:**
  • Model trained on specific population - may not generalize universally
  • Requires complete feature data for accurate predictions
  • Should complement, not replace, clinical judgment
  • Needs validation on external datasets before deployment
  • Regular monitoring for model drift over time

🎯 **Next Steps:**
  1. External validation on independent patient cohorts
  2. Prospective clinical trial to assess real-world impact
  3. Interpretability analysis (SHAP values) for transparency
  4. Cost-effectiveness analysis for healthcare system
  5. Integration with existing clinical workflows
    """)
    
    print("\n" + "=" * 100)
    print("✓ ANALYSIS COMPLETE")
    print("=" * 100)
else:
    print("⚠️  No model results available for analysis")
    print("=" * 100)

### Project Summary and Key Achievements

---

## ✅ **What We Accomplished**

### 1. **Data Pipeline**
   - ✓ Loaded Chronic Kidney Disease dataset from OpenML (dataset #42972)
   - ✓ Comprehensive initial data inspection and quality assessment
   - ✓ Complete statistical profiling of all features

### 2. **Exploratory Data Analysis**
   - ✓ Correlation analysis with heatmap visualization
   - ✓ Feature distribution analysis (histograms and box plots)
   - ✓ Missing value detection and visualization
   - ✓ Class balance assessment and imbalance detection
   - ✓ Outlier identification

### 3. **Data Preprocessing**
   - ✓ Missing value imputation (mean for numerical, mode for categorical)
   - ✓ Categorical variable encoding (Label Encoding)
   - ✓ Stratified train-test split (80-20) maintaining class distribution
   - ✓ Feature scaling (StandardScaler) for neural network optimization
   - ✓ Proper data leakage prevention (fit on train, transform on test)

### 4. **Model Development**
   - ✓ Multi-Layer Perceptron Neural Network with optimized architecture
   - ✓ Two hidden layers (100, 50 neurons) with ReLU activation
   - ✓ Adam optimizer with adaptive learning rate
   - ✓ Early stopping to prevent overfitting
   - ✓ L2 regularization for weight constraint

### 5. **Comprehensive Evaluation**
   - ✓ **Accuracy**: Overall prediction correctness
   - ✓ **Precision**: Positive prediction reliability
   - ✓ **Recall**: Sensitivity to detecting CKD cases
   - ✓ **F1-Score**: Balanced performance metric
   - ✓ **ROC-AUC**: Discriminative ability assessment
   - ✓ **Confusion Matrix**: Detailed prediction breakdown
   - ✓ **5-Fold Cross-Validation**: Robust performance estimation

### 6. **Visualizations**
   - ✓ Performance metrics bar charts
   - ✓ Confusion matrix heatmap
   - ✓ ROC curve with optimal threshold identification
   - ✓ Feature correlation heatmap
   - ✓ Distribution and outlier box plots

---

## 🏥 **Clinical Insights**

### **Why Early CKD Detection Matters:**
- Chronic kidney disease affects millions worldwide
- Often asymptomatic in early stages
- Early intervention can slow or halt progression
- Reduces need for dialysis and transplantation
- Significantly improves patient quality of life
- Reduces healthcare system costs

### **Model's Clinical Value:**
- **Screening Tool**: Identifies at-risk patients for further evaluation
- **Risk Stratification**: Prioritizes patients based on severity
- **Decision Support**: Assists clinicians with data-driven insights
- **Resource Optimization**: Focuses limited resources on highest-risk patients
- **Preventive Care**: Enables early interventions before irreversible damage

### **Prioritizing Recall in Medical Context:**
- **High recall is critical**: Missing a CKD case has severe consequences
- **False negatives are costly**: Delayed treatment leads to progression
- **False positives manageable**: Additional testing has lower risk
- **Balance required**: Need acceptable precision to avoid alarm fatigue

---

## ⚠️ **Limitations and Considerations**

### **Model Limitations:**
1. **Generalizability**: Trained on specific population, may not transfer universally
2. **Data Quality**: Performance depends on complete, accurate input data
3. **Temporal Drift**: Medical knowledge and populations evolve over time
4. **Interpretability**: Neural networks are "black boxes" - hard to explain decisions
5. **Feature Dependencies**: Assumes features available in clinical setting

### **Deployment Considerations:**
1. **Validation Required**: Needs testing on external, independent datasets
2. **Clinical Integration**: Must fit into existing healthcare workflows
3. **Regulatory Approval**: May require FDA or equivalent approval
4. **Continuous Monitoring**: Performance tracking after deployment
5. **Human Oversight**: Should augment, not replace, clinical judgment

---

## 🚀 **Future Work and Enhancements**

### **Model Improvements:**
- [ ] **Ensemble Methods**: Combine multiple models for better performance
- [ ] **Hyperparameter Tuning**: Grid/random search for optimal configuration
- [ ] **Feature Engineering**: Create interaction terms and polynomial features
- [ ] **Deep Learning**: Explore deeper architectures (CNN, RNN for time-series)
- [ ] **Transfer Learning**: Leverage pre-trained medical models

### **Interpretability:**
- [ ] **SHAP Values**: Understand feature contributions to predictions
- [ ] **LIME**: Local interpretability for individual predictions
- [ ] **Attention Mechanisms**: Visualize which features model focuses on
- [ ] **Feature Importance**: Rank features by predictive value

### **Clinical Integration:**
- [ ] **Web Application**: User-friendly interface for clinicians
- [ ] **EHR Integration**: Direct connection to electronic health records
- [ ] **Mobile App**: Point-of-care predictions on tablets/phones
- [ ] **API Development**: Allow integration with existing systems
- [ ] **Real-time Monitoring**: Continuous patient risk assessment

### **Validation Studies:**
- [ ] **External Validation**: Test on datasets from different institutions
- [ ] **Prospective Trial**: Evaluate impact in real clinical settings
- [ ] **Subgroup Analysis**: Performance across demographics (age, gender, ethnicity)
- [ ] **Cost-Effectiveness**: Economic analysis of implementation
- [ ] **Multi-center Study**: Collaborative validation across hospitals

### **Advanced Analytics:**
- [ ] **Longitudinal Analysis**: Track CKD progression over time
- [ ] **Survival Analysis**: Predict time to renal failure
- [ ] **Personalized Thresholds**: Adjust decision boundary per patient
- [ ] **Confidence Estimation**: Provide uncertainty quantification
- [ ] **Active Learning**: Identify cases for expert review to improve model

---

## 📚 **Technical Documentation**

### **Technologies Used:**
- **Python 3.x**: Primary programming language
- **Scikit-learn**: Machine learning library
- **Pandas/NumPy**: Data manipulation and numerical computing
- **Matplotlib/Seaborn**: Data visualization
- **Jupyter Notebook**: Interactive development environment

### **Model Specifications:**
- **Algorithm**: Multi-Layer Perceptron (MLPClassifier)
- **Architecture**: Input → 100 neurons → 50 neurons → Output
- **Activation**: ReLU (hidden layers)
- **Optimizer**: Adam with adaptive learning rate
- **Regularization**: L2 penalty (alpha=0.0001)
- **Training**: Early stopping with 10% validation set

### **Dataset Information:**
- **Source**: OpenML (ID: 42972)
- **Task**: Binary classification (CKD presence/absence)
- **Split**: 80% training, 20% testing (stratified)
- **Preprocessing**: Imputation, encoding, scaling

---

## 🎓 **Project Complete!**

This comprehensive analysis demonstrates the potential of machine learning, specifically Multi-Layer Perceptron neural networks, for chronic kidney disease risk prediction. The model shows promising performance and, with proper validation and clinical integration, could serve as a valuable tool for early detection and intervention.

**Key Takeaway**: Machine learning can augment clinical decision-making, but should always be used responsibly with appropriate validation, monitoring, and human oversight.

---

**BME6938 Spring 2026 - Group 7**  
**Last Updated**: March 4, 2026